<a href="https://www.kaggle.com/code/sunnykumar001/house-loan?scriptVersionId=349492810" target="_blank"><img align="left" alt="Kaggle" title="Open in Kaggle" src="https://kaggle.com/static/images/open-in-kaggle.svg"></a>

In [ ]:
# This Python 3 environment comes with many helpful analytics libraries installed
# It is defined by the kaggle/python Docker image: https://github.com/kaggle/docker-python
# For example, here's several helpful packages to load

import numpy as np # linear algebra
import pandas as pd # data processing, CSV file I/O (e.g. pd.read_csv)

import os
for dirname, _, filenames in os.walk('/kaggle/input'):
    for filename in filenames:
        print(os.path.join(dirname, filename))

import kagglehub

In [ ]:
import kagglehub
path = kagglehub.competition_download('home-credit-default-risk')
print("Path to competition files:", path)

In [ ]:
import os
for dirpath, dirnames, filenames in os.walk(path):
    print(f"Current Directory: {dirpath}")
    print(f"Subdirectories: {dirnames}")
    print(f"Files: {filenames}\n")

In [ ]:
from pathlib import Path
data = Path('/kaggle/input/competitions/home-credit-default-risk/application_train.csv')
train_df = pd.read_csv('/kaggle/input/competitions/home-credit-default-risk/application_train.csv')
train_df.describe()

In [ ]:
import matplotlib.pyplot as plt
import seaborn as sns

train_df["DAYS_EMPLOYED_CLEANED"] = train_df["DAYS_EMPLOYED"].replace(365243, np.nan)
train_df["YEARS_EMPLOYED"] = train_df["DAYS_EMPLOYED_CLEANED"].abs() / 365.25

# Create a clean, readable box plot
plt.figure(figsize=(10, 4))
sns.boxplot(data=train_df, x="YEARS_EMPLOYED", color="skyblue")

# Style the chart
plt.title(
    "Employment Length Distribution (In Years)", fontsize=14, fontweight="bold"
)
plt.xlabel("Years at Current Job", fontsize=12)
plt.grid(axis="x", linestyle="--", alpha=0.5)

plt.tight_layout()
plt.show()

In [ ]:
train_df.drop("DAYS_EMPLOYED_CLEANED",axis=1,inplace=True,errors="ignore")
train_df.drop("DAYS_EMPLOYED",axis=1,inplace=True,errors="ignore")


In [ ]:
train_df.info()

In [ ]:
n_rows , n_columns = train_df.shape

numeric_data = train_df.select_dtypes(include=["number"]).columns
categorical_data = train_df.select_dtypes(include=['object']).columns

print(f"DATAFRAME :{data.name} size :{(data.stat().st_size) / 1024**2:.0f} MB")
print(f"{n_rows} ROWS x {n_columns} COLUMNS")
print(f"NUMERIC : {len(numeric_data)}")
print(f"CATOGORICAL : {len(categorical_data)}")
print(f"SK_ID_CURR : {train_df.SK_ID_CURR.nunique()} "
      f"(unique : {train_df.SK_ID_CURR.is_unique})")

In [ ]:
test_data = Path("/kaggle/input/competitions/home-credit-default-risk/application_test.csv")
test_df = pd.read_csv("/kaggle/input/competitions/home-credit-default-risk/application_test.csv")

In [ ]:
n_rows , n_columns = test_df.shape

numeric_data = test_df.select_dtypes(include=["number"]).columns
categorical_data = test_df.select_dtypes(include=['object']).columns

print(f"DATAFRAME :{test_data.name} size :{(test_data.stat().st_size) / 1024**2:.0f} MB")
print(f"{n_rows} ROWS x {n_columns} COLUMNS")
print(f"NUMERIC : {len(numeric_data)}")
print(f"CATOGORICAL : {len(categorical_data)}")
print(f"SK_ID_CURR : {test_df.SK_ID_CURR.nunique()} "
      f"(unique : {test_df.SK_ID_CURR.is_unique})")

In [ ]:
import pandas as pd 
import numpy as np
from pathlib import Path
from time import time
from zipfile import ZipFile
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.dummy import DummyClassifier
from sklearn.compose import ColumnTransformer
from sklearn.ensemble import (RandomForestClassifier,
                              HistGradientBoostingClassifier,
                              AdaBoostClassifier)
from sklearn.preprocessing import (StandardScaler,
                                   OneHotEncoder)
from sklearn.model_selection import train_test_split
# from sklearn.preprocessing import spl
from sklearn.pipeline import Pipeline
from sklearn.impute import SimpleImputer
from sklearn.metrics import (
    accuracy_score,
    roc_auc_score
)

In [ ]:
TARGET = train_df["TARGET"]
positive_rate = TARGET.mean()
imbalance = (1-positive_rate)/positive_rate

count = TARGET.value_counts().sort_index()
print("Target Distribution")
print(f"On_time payment        : {count[0]:>7,}    {(1-positive_rate)*100:5>.2f}%")
print(f"delayed                : {count[1]:>7,}    {positive_rate*100:5>.2f}% ")
print(f"imablance (ratio diff) : {imbalance:5>.2f}%")

Dummy = DummyClassifier(strategy="most_frequent").fit(train_df["AMT_CREDIT"],TARGET)
predict_prob = Dummy.predict_proba(train_df["AMT_CREDIT"])
acc = Dummy.predict(train_df["AMT_CREDIT"])
rou = Dummy.predict_proba(train_df["AMT_CREDIT"])[:,0]
print(f"Accuracy      : {accuracy_score(TARGET,acc):>6.2f}")
print(f"rou_auc_score : {roc_auc_score(TARGET,rou):>6.2f}")

In [ ]:
plt.figure(figsize=(10, 5))

# Plot overlaying density curves for both target classes
sns.kdeplot(
    data=train_df[train_df["TARGET"] == 0],
    x="YEARS_EMPLOYED",
    label="Repaid on Time",
    shade=True,
    color="g",
)
sns.kdeplot(
    data=train_df[train_df["TARGET"] == 1],
    x="YEARS_EMPLOYED",
    label="Defaulted",
    shade=True,
    color="r",
)

plt.title("How Employment Length Affects Loan Default Risk", fontweight="bold")
plt.xlabel("Years Employed")
plt.ylabel("Density")
plt.legend()
plt.show()

In [ ]:
def merging_df(main_df:pd.DataFrame,
               agg_df,
               by:str):
    final_df = main_df.copy()
    if isinstance(agg_df,list):
        for i in agg_df:
            final_df = main_df.merge(i,on=by,how="left")
            return final_df
    else:
        final_df = main_df.merge(agg_df,on=by,how="left")

        return final_df

In [ ]:
# bureau = pd.read_csv("/kaggle/input/competitions/home-credit-default-risk/bureau.csv")
# bureau_balance = pd.read_csv("/kaggle/input/competitions/home-credit-default-risk/bureau_balance.csv")
# installments_payments = pd.read_csv("/kaggle/input/competitions/home-credit-default-risk/installments_payments.csv")
# previous_application = pd.read_csv("/kaggle/input/competitions/home-credit-default-risk/previous_application.csv")
# pos = pd.read_csv("/kaggle/input/competitions/home-credit-default-risk/POS_CASH_balance.csv")
# credit = pd.read_csv("/kaggle/input/competitions/home-credit-default-risk/credit_card_balance.csv")

In [ ]:
# bureau_df = merging_df(bureau,bureau_balance,by="SK_ID_BUREAU")

In [ ]:
# bureau_df.head()

In [ ]:
# for i in [pos,installments_payments,credit]:
#     print(i.shape)

In [ ]:
# prev = merging_df(previous_application,[pos,installments_payments,credit],by="SK_ID_PREV")

In [ ]:
# prev["SK_ID_CURR"] = prev["SK_ID_CURR_x"]
# prev.drop("SK_ID_CURR_x",axis = 1)

In [ ]:
# train_merged_df = merging_df(train_df,[prev,bureau],by="SK_ID_CURR")
# test_merged_df = merging_df(test_df,[prev,bureau],by="SK_ID_CURR")

In [ ]:
import pandas as pd
def aggregating_data(df:pd.DataFrame,
                         id_colmn : str,
                         df_agg: str|list =None,
                         cat_agg_param=["mean"],
                         cat_colm:bool = False,
                         drop_colmn: str | list = None,
                         if_drop : bool = False):
    final_df = df.copy()
    # if if_drop:
    #     final_df.drop(columns=[drop_colmn], errors='ignore')
    #     print("removed")

    num_df = final_df.select_dtypes(include="number")
    cat_df = final_df.select_dtypes(include="object")
    num_colm_df = num_df.columns.to_list()
    if id_colmn in num_colm_df:
        num_colm_df.remove(id_colmn)
    if drop_colmn in num_colm_df:
        num_colm_df.remove(drop_colmn)
    if cat_colm:
        cat_colm_df = cat_df.columns.to_list()
    else:
        cat_colm_df = False
    print("agg_started")



    if num_colm_df:
        print("grouping started")
        if df_agg is None:
            agg_data = final_df.groupby(id_colmn)[num_colm_df]
        else:
            agg_data = final_df.groupby(id_colmn)[num_colm_df].agg(df_agg)
            agg_data.columns = [f"{c}_{s}".upper() for c,s in agg_data.columns]
            agg_data.reset_index(inplace=True)
        print("num_done")

    if cat_colm_df:
        print("cat_group started")
        dummies = pd.get_dummies(cat_df[cat_colm_df])
        dummies[id_colmn] = final_df[id_colmn]
        cat_agg = dummies.groupby(id_colmn).agg(cat_agg_param)
        cat_agg.columns = [f"{c}_{s}".upper() for c,s in cat_agg.columns]
        cat_agg.reset_index(inplace=True)
        agg_data = agg_data.merge(cat_agg,how="left",on=id_colmn)

    return agg_data

In [ ]:
import pandas as pd
import gc

def aggregate_and_merge(main_df_path, output_name="final_dataset.parquet"):
    print("Starting data assembly...")
    

    main_df = pd.read_csv(main_df_path)
    
    print("Processing Bureau and Bureau Balance...")
    bureau = pd.read_csv("/kaggle/input/competitions/home-credit-default-risk/bureau.csv")
    bureau_balance = pd.read_csv("/kaggle/input/competitions/home-credit-default-risk/bureau_balance.csv")
    print("read csv")
    # Aggregate bureau balance to bureau level first (on SK_ID_BUREAU)
    bb_agg = aggregating_data(bureau_balance,id_colmn="SK_ID_BUREAU",drop_colmn="SK_ID_CURR")
    # bb_agg.columns = ['_'.join(col).strip() for col in bb_agg.columns.values]
    bureau = bureau.merge(bb_agg, on='SK_ID_BUREAU', how='left')
    print("bureau agg done")
    # Aggregate bureau to client level (on SK_ID_CURR)
    bureau_agg = bureau.drop(columns=['SK_ID_BUREAU'])
    
    
    # Merge to main and clear memory
    main_df = main_df.merge(bureau_agg, on='SK_ID_CURR', how='left')
    del bureau, bureau_balance, bb_agg, bureau_agg; gc.collect()

    print("Processing Previous Applications...")
    prev = pd.read_csv("/kaggle/input/competitions/home-credit-default-risk/previous_application.csv")
    prev_agg = aggregating_data(prev,id_colmn="SK_ID_CURR",if_drop=True,drop_colmn='SK_ID_PREV')
    print("done")
    main_df = main_df.merge(prev_agg, on='SK_ID_CURR', how='left')
    del prev, prev_agg; gc.collect()

    for table_name, path in [
        ("POS", "/kaggle/input/competitions/home-credit-default-risk/POS_CASH_balance.csv"),
        ("INSTAL", "/kaggle/input/competitions/home-credit-default-risk/installments_payments.csv"),
        ("CREDIT", "/kaggle/input/competitions/home-credit-default-risk/credit_card_balance.csv")
    ]:
        print(f"Processing {table_name}...")
        child_df = pd.read_csv(path)
        
        # Group directly by client ID, dropping the sub-contract ID to preserve information
        child_agg = aggregating_data(child_df,id_colmn="SK_ID_CURR",if_drop=True,drop_colmn='SK_ID_PREV')
        
        main_df = main_df.merge(child_agg, on='SK_ID_CURR', how='left')
        del child_df, child_agg; gc.collect()

    print(f"Finished! Final Shape: {main_df.shape}")
    return main_df

train_final = aggregate_and_merge("/kaggle/input/competitions/home-credit-default-risk/application_train.csv")
test_final = aggregate_and_merge("/kaggle/input/competitions/home-credit-default-risk/application_test.csv")


In [ ]:
train_final.to_parquet("final_dataset.parquet",index=False)

In [ ]:
test_final.to_parquet("final_test_dataset.parquet",index=False)

In [ ]:
import os

file_path = "final_dataset.parquet"

if os.path.exists(file_path):
    size_mb = os.path.getsize(file_path) / (1024 * 1024)
    print(f"✅ Success! File saved to: {os.path.abspath(file_path)}")
    print(f"File Size: {size_mb:.2f} MB")
else:
    print("Error: File was not found.")

In [ ]:
train_final = pd.read_parquet("final_dataset.parquet")
train_final.info()

In [ ]:
def safely_downcast_df(df):
    start_mem = df.memory_usage().sum() / 1024**2
    print(f"Initial Memory Usage: {start_mem:.2f} MB")
    
    # 1. Downcast Floats to float32
    float_cols = df.select_dtypes(include=['float64']).columns
    df[float_cols] = df[float_cols].astype('float32')
    
    # 2. Safely Downcast Ints to int32 (keeps IDs and large counts intact)
    int_cols = df.select_dtypes(include=['int64']).columns
    df[int_cols] = df[int_cols].astype('int32')
    
    end_mem = df.memory_usage().sum() / 1024**2
    print(f"Final Memory Usage: {end_mem:.2f} MB")
    print(f"Saved {(start_mem - end_mem) / start_mem * 100:.1f}% RAM!")
    
    return df

# Apply it to your data
train_final = safely_downcast_df(train_final)

In [ ]:
X = train_final.drop("TARGET",axis=1)
y = train_final.TARGET

In [ ]:
x_train, x_test, y_train, y_test = train_test_split(X,y,train_size=0.90)

In [ ]:
len(x_train),len(x_test)

In [ ]:
numeric_data = Pipeline(steps=[
    ("SimpleImputer",SimpleImputer(strategy="median")),
    ("scaler",StandardScaler())]
)
# cat_data = Pipeline(steps=[
#     ("imputer", SimpleImputer(strategy="most_frequent",fill_value="missing")),
#     ("one",OneHotEncoder(handle_unknown="ignore",sparse_output=True))
# ])
column_Transformer = ColumnTransformer(transformers=[
    ("num",numeric_data,x_train.select_dtypes(include="number").columns.to_list()),
    # ("cat",cat_data,x_train.select_dtypes(include="object").columns.to_list())
])

whole_process = Pipeline(steps=[
    ("Column_Transformer",column_Transformer),
    ("model",RandomForestClassifier(n_estimators=50))
])
model = whole_process.fit(x_train,y_train)
model

In [ ]:
acc = model.predict(x_test)
rou = model.predict_proba(x_test)[:,1]
print(f"Accuracy      : {accuracy_score(y_test,acc):>6.2f}")
print(f"rou_auc_score : {roc_auc_score(y_test,rou):>6.2f}")